# 3 — Is CEBRA learning a useful cue representation?

This standalone notebook follows a fresh **cue-only CEBRA-Time** experiment from
source interpretation to its optimization, embedding geometry and matched decoder
results. It asks three distinct questions: does optimization behave as intended,
does the representation retain variation, and can fixed downstream readouts recover
endpoint conditions on source-separated development assemblies? The answer to one
need not be the answer to the others. We also test source-budget sensitivity and
an alignment-breaking permutation control.

The package supplies tested operations, while the cells below expose the scientific
choices: what counts as an observation, why conditions are matched, how contributors
are separated, which vectors form contrastive pairs, what a readout measures and
what a three-dimensional view cannot establish. No previous notebook's objects or
historical fitted models are required. The [research walkthrough](00_research_guide.ipynb)
connects this cue assay to the historical work, including joint cue/navigation training.

**Evidence status:** invented inputs and 20 optimizer updates demonstrate the
software; their scores are not biological evidence or a convergence study. Supplied
lab inputs use a separately gated exploratory protocol with 1,000 updates and a
selected source panel. The revised source notebook has empty outputs and is not an
executed result. Existing historical runs belong to their recorded code and cohort.
This revision adds inspection and explanation without changing the fit grid, fitting
another model or authorizing a new study.

In [ ]:
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')
from neuro.notebook_inputs import configuration_from_environment, inspect_inputs, load_inputs, effective_pair_map
from neuro.pseudopopulation import sample_plan, materialize, support
from neuro.notebook_artifacts import ArtifactStore, FitAccounting, target_heads, write_private_json, verify_artifacts, runtime_identity, validate_kernel_identity
from neuro.notebook_support import workspace, usage_summary, source_budget, checked_assemblies, compact_metrics, plot_confusions

configuration = configuration_from_environment()
settings = configuration['execution']
output = workspace()
def private_warning(message, category, filename, lineno, file=None, line=None):
    # Preserve full provenance privately; do not render host paths in shared figures/output.
    warning_path = output / 'warnings-private.jsonl'
    with warning_path.open('a') as handle:
        handle.write(json.dumps({'message': str(message), 'category': category.__name__, 'filename': filename, 'lineno': lineno}) + '\n')
    warning_path.chmod(0o600)
    print(category.__name__ + ': ' + str(message), file=sys.stderr)
warnings.showwarning = private_warning
store = ArtifactStore(output / 'artifacts')
store.save('configuration', configuration)
store.save('kernel_runtime_before_inputs', validate_kernel_identity(configuration))
print('Input mode:', configuration['mode'], '| exploratory computational assay; no biological IID inference')

## Establish what the observations and targets mean

The direct measurements are sorted spike timestamps and recorded task-event markers.
A sorted unit is an inferred spike cluster, not a census of every hippocampal neuron.
For lab inputs, the analysis converts spikes into counts in ten half-open 100-ms
cue bins spanning 0–1 s from cue onset; the smaller invented fixture has four bins.
The marker-derived start and goal are provisional task labels:
agreement between recording streams is a consistency check, not independent proof of
the experiment's semantic meaning. Source eligibility, exclusions and the marker
join must be inspected before interpreting a decoder.

One source trial records only the units in its session block. A zero means an observed
bin containing no spikes; an off-block NaN means that unit was not recorded in that
source. Treating both as zero would manufacture measurements. To form a complete
pseudopopulation row, we match one source from **each block on the known ordered
start→goal pair** and copy that block's units and all bins together. Each resulting
row is assembled, not simultaneously recorded. This preserves within-block observations
while imposing a cross-block combination; it cannot preserve unmeasured cross-session
noise correlations or reveal a real animal's full population state.

This construction is defensible for a **condition-level assay** when every retained
block supports every requested condition in each split, units and timing agree, and
source contributors stay separated. It is not an unknown-label single-trial prediction
pipeline: the condition is needed before constructing the input. Balanced numbers of
assembled rows do not imply balanced numbers of real sources. Sparse block/pair pools
are reused more, some source measurements are never drawn, and complete-support
selection changes the population being studied. Notebook A exposes the selection
waterfall, support matrices, exclusions and source reuse in greater detail; this
notebook independently reconstructs its own inputs and repeats the core gates.

For supplied lab inputs, use only the original outer-training partition. Keep the
historically exposed development assignment frozen, embargo any training example
sharing a current or predecessor contributor with **any original development
example**, then select the complete-support panel. The predecessor matters because
the previous trial contributes the current start label. The historical source route
selected 27 units in 10 sessions; the 195-unit/58-session compact inventory does not
mean that all those units survive this analysis. The printed audit is authoritative
for this execution. Contributor separation still leaves session and longer-range
serial dependence, and development is not a pristine test set.

**Read the next outputs as a gate.** Inspect selected train/development counts,
exclusion reasons, contributor overlap and the source×unit×bin tensor. Missing support,
unexplained selection differences or overlap require investigation before fitting;
they are not reasons to silently repair labels or fill absent neural values.

In [ ]:
inspection = inspect_inputs(configuration)  # Metadata/hash step only for supplied inputs.
store.save('metadata_selection', inspection)  # Protected provenance; never displayed.
print(json.dumps(inspection.audit, sort_keys=True))
assert inspection.audit['contributor_overlap'] == 0
assert inspection.audit['selected_train'] > 0 and inspection.audit['selected_development'] > 0

In [ ]:
data, neural_audit = load_inputs(inspection)  # Supplied neural path remains explicitly review-gated.
store.save('source_data', data)
store.save('neural_audit', neural_audit)
print(json.dumps(neural_audit, sort_keys=True))
assert np.isfinite(data.values[data.observed]).all()
assert np.isnan(data.values[~data.observed]).all()
assert not data.dependency_audit()['overlapping_contributor_ids']
print('Source × unit × bin shape:', data.values.shape, '| units:', data.value_unit, '| clock units:', data.time_unit)

## Freeze the comparison before learning from spikes

We hold the development **values, labels, exposure and contributing sources** fixed
across observed seeds, representations and source budgets. The function
`checked_assemblies` tests this equality, not merely an equal row count. Each observed
training seed changes the constructed training sample (and CEBRA initialization and
sampler when present), while development assembly seed remains zero. These are
computational perturbations, not new subjects or independent biological replications.

The full/reduced intervention asks how sensitive this fixed workflow is to its
available training-source pool. Within every block×condition pool, reduced mode keeps
a deterministic nested `floor(n/2)` subset using the same fixed subset seed. The unit
panel, condition list, bin definitions and synthetic-row budget stay fixed. Every
learned transformation and decoder is refitted on the new training assembly. This
is not deletion of half the neurons, a new development set, or an exact 50% reduction
in independent measurements: floor rounding and source reuse matter. Inspect eligible,
actually used, unused and repeated counts alongside the score changes.

The independent provenance unit is a real source trial **plus its target
contributors**, with trials nested in sessions. Our estimand is descriptive
recoverability of the matched condition in this selected pseudopopulation, under the
declared source split and construction. A higher development score supports a better
readout of these constructed examples. It does not establish trial-specific path
prediction, intention, causal mechanism or generalization to a new animal/session.

## What is CEBRA-Time being asked to learn here?

CEBRA is a contrastive encoder: the sampling rule specifies which neural vectors
should be similar, and an optimization objective rewards those similarities relative
to negatives. Time-only and behavior-conditioned sampling are different scientific
choices; the name CEBRA alone does not specify one. The original method is described
by [Schneider, Lee and Mathis (2023)](https://www.nature.com/articles/s41586-023-06031-6).
The statements below describe **this repository's cue-only implementation**, whose
sampler and normalization must be reviewed directly.

An input vector contains all U standardized units at one native cue bin. Select an
assembled training row uniformly, then select one of its forward adjacent-bin pairs
uniformly: (0,1), (1,2), …, (T−2,T−1). The first vector is the reference and the next
is its positive. Draw each negative from a uniformly selected row and bin. Pairing
does not require negatives to have a different endpoint label: same-condition
negatives are possible. Positive pairing
never crosses a row or jumps from its last bin to the next row's first bin. In this
cue-only route phase probability is one for cue; the logged navigation exposure must
be zero. The generic sampler also supports historical joint-phase work, but that
default is overridden here. References cannot occupy the last bin and positives
cannot occupy the first, so their exposure histograms need not look identical.

The encoder is the existing `offset1-model`, with eight outputs, fixed cosine
InfoNCE temperature 1 and Adam. It transforms a single native unit vector; the
temporal structure is supplied by **which vectors are paired**, not by feeding the
entire cue as one encoder input. Conceptually the loss rewards similarity between
the reference and positive while penalizing similarity to sampled negatives through
a log-sum-exp contrast. The implementation logs total, positive and negative loss
terms and temperature; their absolute values depend on the objective and sampling,
so compare trends within a matched configuration, not with classifier log loss.
The exact solver is in [cebra_solver.py](../src/neuro/cebra_solver.py), and
the audited native-pair construction is in [cebra.py](../src/neuro/cebra.py).

For reference rᵢ, positive pᵢ and a sampled negative bank nⱼ, its objective is
the batch mean of −sim(f(rᵢ), f(pᵢ))/τ + log Σⱼ exp(sim(f(rᵢ), f(nⱼ))/τ),
with τ=1 and cosine similarity. The positive is not explicitly appended to this
denominator; a source/vector can recur through sampling. The sampler therefore
defines the learning problem as much as the network architecture does.

The Time sampler does not use goal equality to select positives; it uses within-row
adjacency. Nonetheless **the input rows were already constructed using known pair
labels**, so this pipeline is not label-free in the broader experimental sense.
`train_targets` annotate rows and support an assertion that positive pairing stays
within their labels; they are not a supervised goal loss. An NLP analogy is learning
from adjacency in a corpus whose documents were deliberately assembled by label:
an unsupervised objective does not erase the selection mechanism of its inputs.

Normalization is fitted only on pooled training row/bin vectors, then frozen.
CEBRA and PCA transform each bin to eight coordinates, concatenate them in
time-major order, and feed a separate training-scaled logistic head for pair/start/goal.
Raw features are unit-major. All three readouts see the whole cue and identical
development lineage. The readout is a second learning stage; a contrastive encoder
can optimize successfully while its whole-cue features carry little information useful
to this particular linear classifier.

## Keep observed and permuted worlds separate

Observed seeds 0 and 1 each run full and reduced sources. The alignment-breaking
control uses computational seed 0 and the same two source budgets, for six CEBRA,
six PCA and 54 logistic fits in total. In the control, joint **pair labels** are
permuted at the real-source level within each retained block×split **before**
matching. This keeps the observed provenance intact while changing the effective
condition used to assemble rows and train readouts. Permuting only the already
assembled row labels would test a different mechanism.

Each world has its own fixed development assembly. Within that world, the
full/reduced comparison has identical development values and lineage; between
worlds, construction itself changes. Label permutation preserves block/split
label counts but does not make serially dependent biological trials exchangeable.
One permutation is therefore an **alignment-breaking diagnostic**, not a calibrated
null distribution or a p-value. A residual score can reflect finite samples,
structured dependence or the construction; it cannot be declared significant.

The source-budget intervention complements this control. It asks whether a
reported score depends strongly on available real training sources while holding
the synthetic-row count fixed. More synthetic rows can improve numerical sampling
of an existing pool without adding independent measurements. Read source usage,
losses, rank and the scores together, and keep world/seed/budget in every table and
caption. Do not average the observed and permuted results into a single headline.

In [ ]:
import neuro.cebra as cebra
validate_kernel_identity(configuration)  # Includes actual optional dependency origins/code before fitting.
from neuro.pca_lda import fit_standardizer, raw_features, fit_pca
from neuro.notebook_support import fit_logistic
assert settings['seeds'] == [0, 1]
print({k: settings[k] for k in ('iterations', 'batch_size', 'hidden', 'output_dimension', 'learning_rate')})
print('Native positives:', [[b, b + 1] for b in range(data.values.shape[2] - 1)], '| references row-uniform; negatives row/bin-uniform')
accounting = FitAccounting(cebra=True).start()
results = []
from neuro.notebook_diagnostics import metrics_table, plot_decoder_summary, embedding_summary, plot_embedding_views, learning_summary, plot_learning
case_diagnostics = {}
reference_view = None

## Is the model learning? Separate optimization, geometry and readout transfer

There are three questions to inspect for **every** fitted case. First, does the
declared optimization run finish with finite losses and parameters, the expected
number of updates, and the intended sampler exposure? Look at total, positive and
negative terms together. A falling total loss means the model is improving its
training contrastive objective; stochastic curves need not decrease at every update.
It does not measure held-out endpoint recovery. The demo's **20 updates** are a
software exercise, not a convergence experiment; the reviewed real configuration
fixes **1,000 updates**. Neither budget can be optimized by inspecting development
scores during this notebook, and this revision does not execute another fit.

Second, has the representation become degenerate? Inspect coordinate standard
deviations, the coordinate-covariance eigenvalue spectrum and descriptive effective
rank of pooled embeddings on training and development. Very small variation or
near-rank-one features can explain an apparently uninformative point cloud; an
eight-output network does not guarantee eight useful dimensions. Effective rank
here is the covariance participation ratio, (sum of eigenvalues)² / sum of squared
eigenvalues; it describes concentration of variance, not a count of biological
states. Treat it as a diagnostic rather than a pass/fail biological test. Broadly
distributed variance also does not guarantee useful task information.
The sampled-vector use histogram describes optimization exposure, not the number of
independent recorded trials; resampling a source does not create new neural evidence.

Third, do fixed, matched development readouts improve relative to raw/PCA and the
uniform references, and which classes account for that result? Use all seeds,
confusions, source-budget comparisons and the train/development gaps. A falling loss
with no readout improvement means the temporal objective improved without a matching
gain in this endpoint assay. A good readout with a large gap invites scrutiny of
overfitting and source composition. A clear 3D cluster on its own answers neither
question. There is no held-out contrastive-loss curve, checkpoint accuracy sweep or
early-stopping study in this grid; the final readouts cannot establish when performance
emerged or which update budget is optimal.

## Decide what a successful decoder would mean

The decoder sees the entire cue, with bins concatenated in order; it does not decode
each time bin separately. We fit **three independent classification heads**: ordered
pair, start, and goal. Pair classification asks which joint endpoint condition was
used to assemble a row; start/goal ask its components. None asks which ordered path
the animal took beyond those endpoints. A strong goal head can coexist with a weak
pair head, and an easy marginal target must not be presented as successful path decoding.

For K supported classes, balanced accuracy is the mean of the K class recalls.
Accuracy counts all correct rows equally; macro-F1 averages each class's precision/recall
tradeoff. With truly equal evaluation support, accuracy and balanced accuracy coincide;
we still show both to expose support mistakes and make the metric definition explicit.
Log loss is the average negative log probability assigned to the correct class, so
it can worsen when a model becomes confidently wrong even if hard accuracy improves.
Uniform-probability references are **1/K balanced accuracy** and **ln(K) log loss**.
They are descriptive references, not significance cutoffs; macro-F1 has no general
1/K reference. See the [official metric definitions](https://scikit-learn.org/stable/modules/model_evaluation.html).

Always read count **and row-normalized** confusion matrices. Rows are true classes,
columns predicted classes; the diagonal of the normalized matrix is per-class recall.
A concentrated prediction column reveals a decoder favoring one label, even if a
single average looks acceptable. The matrices describe constructed rows, not independent
source-trial counts. The training-minus-development balanced-accuracy gap is a
diagnostic of fitting versus transfer within this assay: a large positive gap invites
an overfitting/distribution-shift explanation; a small gap with low scores is not success.

Compare representations on the same development rows, head, seed and source budget.
Compare raw logistic with PCA/CEBRA logistic to assess the consequences of the
representation under this readout choice; compare raw LDA with raw logistic to assess
the classifier assumption. Fixed regularization is a controlled workflow choice, not
proof that every representation has its optimal decoder. Do not choose the best seed
or hyperparameter after looking at development and call that a held-out result.
The figures below show individual cases and references; no confidence interval or
biological p-value is inferred from pseudorows or the two seeds.

In [ ]:
for world in ('observed', 'permuted'):
    world_map = effective_pair_map(data, permutation_seed=None if world == 'observed' else settings['permutation_seed'])
    development_plan = sample_plan(data, split_id='development', rows_per_condition=settings['development_rows_per_condition'], seed=0, effective_conditions=world_map)
    reference_development = materialize(data, development_plan)
    store.save(world + '_fixed_development', reference_development)
    for seed in (settings['seeds'] if world == 'observed' else [0]):
        for budget in ('full', 'reduced'):
            selected, effective = source_budget(data, world_map, reduced=budget == 'reduced', subset_seed=settings['subset_seed'])
            train, development = checked_assemblies(selected, effective, development_plan, reference_development, config=settings, seed=seed)
            key = f'{world}_seed{seed}_{budget}'
            store.save_bundle(key, train, development)
            print(key, 'source accounting:', json.dumps(usage_summary(selected, train), sort_keys=True))
            standardizer = fit_standardizer(train.values)
            raw_train, raw_dev = raw_features(standardizer.transform(train.values)), raw_features(standardizer.transform(development.values))
            pca = fit_pca(train.values, n_components=settings['pca_components'])
            remaining = float(os.environ.get('NEURO_DEADLINE_MONOTONIC', time.monotonic() + 120)) - time.monotonic()
            encoder = cebra.fit(train.values, train.targets, iterations=settings['iterations'], batch_size=settings['batch_size'],
                                output_dimension=settings['output_dimension'], hidden=settings['hidden'], learning_rate=settings['learning_rate'],
                                seed=seed, max_seconds=remaining, device='cpu')
            store.save_encoder(key + '_raw', standardizer, kind='raw')
            store.save_encoder(key + '_pca', pca, kind='pca')
            store.save_encoder(key + '_cebra', encoder, kind='cebra')
            feature_pairs = {'raw': (raw_train, raw_dev), 'pca': (pca.transform(train.values), pca.transform(development.values)),
                             'cebra': (encoder.transform(train.values), encoder.transform(development.values))}
            print(key, 'Tensor/feature shapes:', {'row_unit_bin': train.values.shape,
                  **{name: pair[0].shape for name, pair in feature_pairs.items()}})
            learned = learning_summary(encoder)
            geometry = {name: embedding_summary(*feature_pairs[name], n_bins=train.values.shape[2])
                        for name in ('pca', 'cebra')}
            case_diagnostics[key] = {'optimization': learned, 'geometry': geometry}
            print(key, 'Optimization:', json.dumps(learned, sort_keys=True))
            print(key, 'Geometry:', json.dumps(geometry, sort_keys=True))
            if key == 'observed_seed0_full':
                reference_view = {'key': key, 'n_bins': train.values.shape[2],
                                  'targets': target_heads(development),
                                  'features': {name: tuple(x.copy() for x in feature_pairs[name])
                                               for name in ('pca', 'cebra')}}
            run_results = []
            for representation, (train_x, dev_x) in feature_pairs.items():
                for head, y in target_heads(train).items():
                    model = fit_logistic(train_x, y, C=settings['logistic_C'], max_iter=settings['logistic_max_iter'])
                    row = store.readout(key + '_' + representation + '_' + head, model, encoder=key + '_' + representation,
                                        bundle=key, head=head, train_features=train_x, development_features=dev_x)
                    run_results.append(row)
            results.extend(run_results)
            print(json.dumps(compact_metrics(run_results), sort_keys=True))
            print('Realized sampler phase exposure:', encoder.diagnostics['exposure_summary']['reference_phase_counts'])
            assert len(encoder.diagnostics['loss']['total']) == settings['iterations']
            assert np.isfinite(encoder.diagnostics['loss']['total']).all()
            fig = plot_learning(encoder, title=key + ': training objective and realized exposure')
            display(fig); plt.close(fig)
            fig = plot_confusions(run_results, title=key + ': development counts and row-normalized confusions')
            display(fig); plt.close(fig)
store.save('aggregate_metrics', results)
store.save('diagnostic_summaries', case_diagnostics)

## Read the matched decoder evidence

The summary keeps observed and permuted worlds distinct and names every seed/source
budget. Compare raw, PCA and CEBRA within one case before comparing budgets. The
printed statements describe the scores just computed; they do not automatically
interpret a difference as learning, significance or biological information. There
are no decoder accuracy-versus-update curves in this grid: each readout uses the
final representation. Revisit the optimization and rank summaries for cases where
the readout result does not match the visual impression.

In [ ]:
# Summaries use already-computed predictions; no model is fitted in this cell.
from html import escape
from IPython.display import HTML
metric_rows = metrics_table(results)
columns = [('world', 'World'), ('seed', 'Seed'), ('budget', 'Sources'), ('model', 'Model'),
           ('head', 'Target'), ('classes', 'K'), ('train_BA_percent', 'Train BA %'),
           ('development_BA_percent', 'Development BA %'), ('uniform_BA_percent', 'Uniform BA %'),
           ('gap_percentage_points', 'Gap (pp)'), ('accuracy_percent', 'Accuracy %'),
           ('macro_F1', 'Macro F1'), ('log_loss_nats', 'Log loss (nats)'),
           ('uniform_log_loss_nats', 'Uniform loss (nats)')]
def metric_text(value):
    return f'{value:.3f}' if isinstance(value, (float, np.floating)) else str(value)
header = '<tr>' + ''.join('<th>' + escape(label) + '</th>' for _, label in columns) + '</tr>'
body = ''.join('<tr>' + ''.join('<td>' + escape(metric_text(row[key])) + '</td>'
                              for key, _ in columns) + '</tr>' for row in metric_rows)
display(HTML('<div style="overflow-x:auto"><table><caption>Matched whole-cue readouts; '
             'constructed rows, no biological uncertainty intervals</caption><thead>'
             + header + '</thead><tbody>' + body + '</tbody></table></div>'))
fig = plot_decoder_summary(results)
display(fig); plt.close(fig)
print('How to read this execution:')
for row in results:
    score = row['development']
    k = len(score['classes'])
    ba = score['balanced_accuracy']
    gap = row['balanced_accuracy_gap']
    relation = 'above' if ba > 1 / k else ('below' if ba < 1 / k else 'at')
    print(f"{row['name']}: development BA {ba:.3f} is {relation} uniform {1 / k:.3f}; "
          f"train minus development {gap:+.3f}; log loss {score['log_loss']:.3f} vs uniform {np.log(k):.3f}.")
print('These are constructed-row summaries, without biological significance or independence claims.')
by_name = {row['name']: row for row in results}
example = by_name['observed_seed0_full_raw_pair']['development']
counts = np.asarray(example['confusion_counts'], dtype=float)
class_support = counts.sum(axis=1)
assert np.all(class_support > 0)
recall = np.diag(counts) / class_support
worked_ba = recall.mean()
worked_accuracy = np.trace(counts) / counts.sum()
np.testing.assert_allclose(worked_ba, example['balanced_accuracy'])
np.testing.assert_allclose(worked_accuracy, example['accuracy'])
print('Worked confusion arithmetic: observed seed 0 / full / raw logistic / pair')
print('True-class supports:', class_support.astype(int).tolist())
print('Diagonal / true-class support = recalls:', np.round(recall, 3).tolist())
print(f'Mean class recall = BA {worked_ba:.3f}; diagonal sum / all rows = accuracy {worked_accuracy:.3f}.')
worst = int(np.argmin(recall))
print('Lowest recall class (first on ties):', example['classes'][worst], f'{recall[worst]:.3f}')
print('Matched source-budget sensitivity (full minus reduced development BA):')
for name, full in by_name.items():
    if '_full_' in name:
        reduced = by_name[name.replace('_full_', '_reduced_', 1)]
        delta = full['development']['balanced_accuracy'] - reduced['development']['balanced_accuracy']
        print(f"{name}: {delta:+.3f}; same development rows within this world/seed.")
print('Inspect the source reuse reports and confusion rows before attributing a difference to representation quality.')

## A 3D distribution view, with an explicit provenance

The displayed case is preselected: **observed seed 0, full sources**. We retain only
that case's train/development feature arrays for this illustration; all six cases
already have saved numerical geometry and learning summaries. The development plots
use its frozen CEBRA/PCA models, never the final permuted reduced model left in a loop
variable. The first-bin panels show coordinates 1–3, colored by start and goal, with
gray training points for reference; pair information has its own decoder head.
The companion all-bin view colors the same coordinates by native cue-bin index.
No extra projection, dimensionality search or supervised plotting model is fitted.

PCA axes are ordered by training variance. CEBRA coordinates are not ordered by
task relevance and their orientation is not a biological coordinate system; coordinate
1 from another fit is not automatically the same axis. A slice of three coordinates
from an eight-dimensional representation can conceal readout information. Points
are constructed population rows (or bins of those rows in the time view), not
simultaneously observed whole-population trials. Apparent separation by endpoint is
descriptive because endpoint matching helped construct the input. Apparent temporal
order may reflect adjacent-bin sampling, common trial timing or source reuse.

Interpret the PCA and CEBRA views alongside their matched scores, not by deciding
which cloud looks prettier. Tight clouds, smooth paths and high numerical rank do
not identify a neural-state manifold, prove dynamics or establish intention. We
deliberately avoid connecting independent synthetic rows with lines: there is no
measured population trajectory between them. Units remain arbitrary standardized
embedding coordinates. The plot is an inspection surface for distribution, spread,
outliers and color organization; the quantitative assay remains the declared readout.

In [ ]:
assert reference_view is not None and reference_view['key'] == 'observed_seed0_full'
print('Displayed case:', reference_view['key'])
for representation in ('pca', 'cebra'):
    train_x, development_x = reference_view['features'][representation]
    fig = plot_embedding_views(train_x, development_x, reference_view['targets'],
                               n_bins=reference_view['n_bins'],
                               title='Observed seed 0 / full sources / ' + representation.upper())
    display(fig); plt.close(fig)
print('Per-case optimization and geometry diagnostics cover:', sorted(case_diagnostics))
print('Grid summary: 6 CEBRA, 6 PCA, 54 logistic; observed and permuted worlds remain separate.')

## How did joint cue/navigation training change the historical result?

**This cue-only notebook cannot estimate the joint-training effect.** The compact
source contract reconstructs cue activity and omits dense navigation behavior.
Reusing a cue-only loss curve or adding a second target head would not create a joint
training comparison. “Joint pair target” (start together with goal) and “joint
cue/navigation training” (two task phases) are different uses of the word joint.

In the historical endpoint study, the same encoder was trained using cue and
navigation vectors, with the joint Time sampler allocating phase draws equally
between cue and navigation. Positives were adjacent native bins **within a phase and
row**; sharing encoder weights did not impose a cross-phase positive pair or align
the phases biologically. At a fixed number of updates, joint training also reallocates
sampling exposure away from cue, so any score difference bundles a change in training
distribution and phase exposure. It is not a pure test of “more neural data.”

The accepted October 7 endpoint evidence used the historical **195-unit/58-session**
panel, three seeds, eight-dimensional 1,000-update CEBRA-Time encoders, and matched
cue readouts. Joint training improved some endpoint scores but did not uniformly
beat raw/PCA references, and the whole-cue arm-gain controls did not establish robust
joint superiority. That is an endpoint result, not evidence that cue and navigation
have the same geometry, that one representation rotates into the other, or that path
information beyond start/goal was recovered. The present complete-support panel and
predecessor embargo differ, so comparing its score directly with the historical joint
score would confound the cohort and construction with training phase.

The [research walkthrough](00_research_guide.ipynb) places the actual historical
figures and their provenance in the full story. For the accepted original numbers,
seed detail, nulls and confusions, use the
[historical endpoint report](../docs/reports/cebra-endpoint-time-readout-20261006/README.md)
and the [saved-model review notebook](../report/endpoint_walkthrough.ipynb).
Those artifacts own their execution evidence; this notebook does not replay them.
A new matched joint experiment would need navigation inputs, an explicit scientific
protocol and fresh execution approval. It is a research design question, not a
missing plotting option in the current cue-only workflow.

## Interpretation limits
These are source-safe conditional pseudopopulations, not simultaneous trials. Matching uses known labels; this is not an unknown-trial prediction pipeline. Bins, synthetic rows and computational seeds are dependent. Place-selected units, provisional marker meaning, inherited preprocessing and historical exposure remain limitations. Contributor separation does not remove long-range serial/session confounds. No biological confidence intervals, p-values, mechanism, dynamics or prospective-path claim follows from these figures. The permutation diagnostic is alignment-breaking, not a biological exchangeability test.

## What the next researcher should be able to defend

**Source trial** means a recorded trial, with a predecessor contributor when required
for label derivation. **Block** is the group of units copied from one recording
session. **Pseudorow** is the condition-matched assembly across blocks; its count is
not biological n. **Representation/encoder** maps neural vectors to features;
**readout/decoder** predicts an operational target from those features. **Development**
means the fixed, historically exposed evaluation assignment here, not an untouched
test set. **Observability in this notebook** means that decisions, intermediate
quantities, failures and provenance can be inspected. **Formal state observability**
would require a specified dynamical system and measurement model showing which
hidden states can be distinguished; classification accuracy does not provide that.

Before handing off a result, write an evidence paragraph containing its target,
source cohort, synthetic counts, split/embargo, time window, representation/readout,
score relative to a named reference, seed and source-budget sensitivity, and strongest
allowed claim. A useful conclusion can be negative: “the encoder's objective improved,
but the matched decoder did not improve over the raw reference” is a complete research
observation. Avoid changing “not established” into evidence that hippocampal coding
is absent. Historical H1 concerns path information beyond endpoints and reproducible
organization; these endpoint heads do not settle it.

As a reading exercise, trace one tensor column back to unit and cue bin, then trace
one assembled row back to its block contributors without exporting their identities.
Explain why class balance in that row table can coexist with skewed source reuse.
Choose one head and reconcile its mean score with the worst recall in its confusion
matrix. Explain why the full/reduced difference is a source-sensitivity result and
why a change in a three-axis display is not automatically a readout improvement.
Finally, list which claims remain impossible even if every software assertion passes.
These exercises require inspecting existing outputs, not running additional models.

The final cell closes fit accounting, authenticates this execution's artifact manifest
and replays its saved transformations/predictions with fitting blocked. Read its
status as a computational check. A defensible scientific result additionally needs
the source/label reasoning above and an independent review of leakage, dependence,
confounds and the written claim. This source notebook's outputs are intentionally
cleared; the revised figures are specifications until an authorized execution produces
them. Earlier successful runs attest their recorded code, not these newly added cells.

In [ ]:
fit_counts = accounting.stop()
store.save('kernel_runtime_final', runtime_identity())
expected_counts = {'pca': 6, 'logistic': 54, 'lda': 0, 'cebra': 6, 'optimizer_updates': 6 * settings['iterations']}
manifest_hash = store.finish(fit_counts=fit_counts, expected_counts=expected_counts)
replay = verify_artifacts(output / 'artifacts', expected_manifest_hash=manifest_hash)
write_private_json(output / 'completion.json', {'manifest_sha256': manifest_hash, 'fit_counts': fit_counts, 'replay': replay})
print(json.dumps(replay, sort_keys=True))
assert replay['status'] == 'fit-blocked-loaded-model-replay-passed'